In [ ]:
from pathlib import Path
import hashlib
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, UnidentifiedImageError

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/raw")

print("Raw data directory:")
print(DATA_DIR.resolve())

print("\nContents:")
for item in DATA_DIR.iterdir():
    print(item)

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

image_paths = [
    p for p in DATA_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

print(f"Total images: {len(image_paths):,}")

In [ ]:
classes = sorted({
    p.parent.name
    for p in image_paths
})

print(f"Number of classes: {len(classes)}")

for i, cls in enumerate(classes, start=1):
    print(f"{i:2d}. {cls}")

In [ ]:
df = pd.DataFrame({
    "path": [str(p) for p in image_paths],
    "label": [p.parent.name for p in image_paths],
})

df.head()

In [ ]:
df.info()

In [ ]:
class_counts = df["label"].value_counts().sort_values(ascending=False)

print(class_counts)

In [ ]:
plt.figure(figsize=(12, 8))

class_counts.plot(kind="bar")

plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
stats = class_counts.describe()

print(stats)

In [ ]:
print("Largest class:", class_counts.max())
print("Smallest class:", class_counts.min())
print("Max/Min ratio:", round(class_counts.max() / class_counts.min(), 2))

In [ ]:
random.seed(42)

sample_classes = random.sample(
    classes,
    min(9, len(classes))
)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.flatten()

for ax, cls in zip(axes, sample_classes):
    class_images = df[df["label"] == cls]["path"].tolist()
    image_path = random.choice(class_images)

    try:
        image = Image.open(image_path).convert("RGB")
        ax.imshow(image)
        ax.set_title(cls, fontsize=9)
    except Exception as e:
        ax.set_title(f"ERROR: {cls}")

    ax.axis("off")

for ax in axes[len(sample_classes):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
dimensions = []
channels = []
corrupt_files = []

for path in image_paths:
    try:
        with Image.open(path) as img:
            dimensions.append(img.size)
            channels.append(img.mode)

    except (UnidentifiedImageError, OSError):
        corrupt_files.append(str(path))

print(f"Corrupt/unreadable images: {len(corrupt_files)}")

In [ ]:
dimension_counts = pd.Series(dimensions).value_counts()

print("Most common image dimensions:")
print(dimension_counts.head(20))

In [ ]:
print(pd.Series(channels).value_counts())

In [ ]:
print("Missing paths:", df["path"].isna().sum())
print("Missing labels:", df["label"].isna().sum())
print("Empty labels:", (df["label"].str.strip() == "").sum())

In [ ]:
def file_hash(path, chunk_size=8192):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)

    return h.hexdigest()


hashes = []

for path in image_paths:
    try:
        hashes.append(file_hash(path))
    except OSError:
        hashes.append(None)

df["sha256"] = hashes

In [ ]:
duplicate_count = df["sha256"].duplicated().sum()

print("Exact duplicate images:", duplicate_count)

In [ ]:
duplicates = df[df["sha256"].duplicated(keep=False)].sort_values("sha256")

duplicates.head(20)

In [ ]:
df["file_size_kb"] = [
    Path(path).stat().st_size / 1024
    for path in df["path"]
]

df["file_size_kb"].describe()

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(df["file_size_kb"], bins=50)

plt.title("Image File Size Distribution")
plt.xlabel("File Size (KB)")
plt.ylabel("Number of Images")

plt.show()

In [ ]:
df["width"] = [size[0] for size in dimensions]
df["height"] = [size[1] for size in dimensions]

df[["width", "height"]].describe()

In [ ]:
plt.figure(figsize=(10, 5))

plt.scatter(df["width"], df["height"], alpha=0.3)

plt.title("Image Resolution Distribution")
plt.xlabel("Width")
plt.ylabel("Height")

plt.show()

In [ ]:
N_SAMPLES_PER_CLASS = 5

for cls in classes:
    samples = df[df["label"] == cls].sample(
        min(N_SAMPLES_PER_CLASS, len(df[df["label"] == cls])),
        random_state=42
    )

    fig, axes = plt.subplots(1, len(samples), figsize=(15, 3))

    if len(samples) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, samples.iterrows()):
        image = Image.open(row["path"]).convert("RGB")
        ax.imshow(image)
        ax.axis("off")

    fig.suptitle(cls)
    plt.tight_layout()
    plt.show()

In [ ]:
duplicate_groups = (
    df[df["sha256"].duplicated(keep=False)]
    .sort_values("sha256")
)

print(f"Duplicate images: {len(duplicate_groups)}")
display(duplicate_groups[["label", "path", "sha256"]])

In [ ]:
duplicate_label_check = (
    duplicate_groups
    .groupby("sha256")["label"]
    .nunique()
)

print("Duplicate groups with different labels:")
print(duplicate_label_check[duplicate_label_check > 1])

In [ ]:
df_clean = df.drop_duplicates(subset="sha256", keep="first").copy()

print("Original images:", len(df))
print("After removing exact duplicates:", len(df_clean))
print("Removed:", len(df) - len(df_clean))

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.20,
    stratify=df_clean["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

In [ ]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
split_distribution = pd.DataFrame({
    "All": df_clean["label"].value_counts(normalize=True),
    "Train": train_df["label"].value_counts(normalize=True),
    "Validation": val_df["label"].value_counts(normalize=True),
    "Test": test_df["label"].value_counts(normalize=True),
}).fillna(0)

display(split_distribution)

In [ ]:
split_distribution.plot(
    kind="bar",
    figsize=(14, 7)
)

plt.title("Class Distribution Across Splits")
plt.xlabel("Class")
plt.ylabel("Proportion")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
train_hashes = set(train_df["sha256"])
val_hashes = set(val_df["sha256"])
test_hashes = set(test_df["sha256"])

print("Train ∩ Validation:", len(train_hashes & val_hashes))
print("Train ∩ Test:", len(train_hashes & test_hashes))
print("Validation ∩ Test:", len(val_hashes & test_hashes))

In [ ]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)

train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/validation.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

In [ ]:
split_counts = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).fillna(0).astype(int)

split_counts["total"] = (
    split_counts["train"]
    + split_counts["validation"]
    + split_counts["test"]
)

display(split_counts)

In [ ]:
split_counts[["train", "validation", "test"]].plot(
    kind="bar",
    figsize=(14, 7)
)

plt.title("Class Counts Across Train / Validation / Test")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:

from pathlib import Path
import sys

ROOT = Path.cwd().parent

print("Notebook working directory:", Path.cwd())
print("Project root:", ROOT)
print("src exists:", (ROOT / "src").is_dir())
print("greenlens exists:", (ROOT / "src" / "greenlens").is_dir())
print("dataset.py exists:", (ROOT / "src" / "greenlens" / "data" / "dataset.py").is_file())
print("Python:", sys.executable)


In [ ]:

from pathlib import Path
import sys

ROOT = Path.cwd().parent.resolve()
SRC_DIR = ROOT / "src"

if not (SRC_DIR / "greenlens").is_dir():
    raise FileNotFoundError(
        f"Could not find the package directory: {SRC_DIR / 'greenlens'}"
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Added to Python path:", SRC_DIR)


In [ ]:

from pathlib import Path
import ast

ROOT = Path.cwd().parent.resolve()
architecture_path = ROOT / "src" / "greenlens" / "models" / "architecture.py"

print("File:", architecture_path)
print("\n--- File contents ---")
print(architecture_path.read_text(encoding="utf-8"))

print("\n--- Defined functions and classes ---")
tree = ast.parse(architecture_path.read_text(encoding="utf-8"))

for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        print(type(node).__name__, node.name)


In [ ]:
import importlib
import greenlens.models.architecture as architecture

print("File:", architecture.__file__)
print("Has build_model:", hasattr(architecture, "build_model"))

importlib.reload(architecture)
print("After reload:", hasattr(architecture, "build_model"))
print("Functions:", [x for x in dir(architecture) if "model" in x.lower()])

In [ ]:
from greenlens.models.architecture import build_model
from greenlens.data.dataset import PlantDiseaseDataset
from greenlens.data.preprocessing import get_transforms

model = build_model(num_classes=15, pretrained=False)
print("Output classes:", model.fc.out_features)

In [ ]:

from greenlens.models.architecture import build_model
from greenlens.data.dataset import PlantDiseaseDataset
from greenlens.data.preprocessing import get_transforms

model = build_model(num_classes=15, pretrained=False)

print("Architecture:", type(model).__name__)
print("Output classes:", model.fc.out_features)
print("Dataset and preprocessing imports: OK")


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent.resolve()
train_df = pd.read_csv(ROOT / "data" / "processed" / "train.csv")

sample_path = Path(train_df.iloc[0]["path"])

print("CSV rows:", len(train_df))
print("Sample path:", sample_path)
print("Exists as written:", sample_path.exists())
print("Exists under ROOT:", (ROOT / sample_path).exists())
print("Current directory:", Path.cwd())
